In [1]:
import sys
sys.path.append("/home/suvendu/mlbd/project1/CSL7100-Project/code")  # adjust path if needed

from etl_new import run_etl

In [2]:
import etl_new
import importlib

importlib.reload(etl_new)

<module 'etl_new' from '/home/suvendu/mlbd/project1/CSL7100-Project/code/etl_new.py'>

In [3]:
from config import print_config
from etl_new import run_etl

print_config()
run_etl()

🔧 CONFIGURATION
PROJECT_ROOT: /home/suvendu/mlbd/project1/CSL7100-Project
USE_HDFS: False
BASE_PATH: file:///home/suvendu/mlbd/project1/CSL7100-Project
PATHS:
  raw: file:///home/suvendu/mlbd/project1/CSL7100-Project/raw
  parquet: file:///home/suvendu/mlbd/project1/CSL7100-Project/parquet
  graph: file:///home/suvendu/mlbd/project1/CSL7100-Project/parquet/graph
  checkpoint: file:///home/suvendu/mlbd/project1/CSL7100-Project/checkpoints
  spark_temp: file:///home/suvendu/mlbd/project1/CSL7100-Project/spark-temp
💻 Running in LOCAL mode


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/21 19:06:18 WARN Utils: Your hostname, suvendu2023, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/21 19:06:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 19:06:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


📥 Loading data from: file:///home/suvendu/mlbd/project1/CSL7100-Project/raw


📊 Schema:
root
 |-- asin: string (nullable = true)
 |-- image: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- overall: double (nullable = true)
 |-- reviewText: string (nullable = true)
 |-- reviewTime: string (nullable = true)
 |-- reviewerID: string (nullable = true)
 |-- reviewerName: string (nullable = true)
 |-- style: struct (nullable = true)
 |    |-- Format:: string (nullable = true)
 |    |-- Shape:: string (nullable = true)
 |    |-- Size:: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- unixReviewTime: long (nullable = true)
 |-- verified: boolean (nullable = true)
 |-- vote: string (nullable = true)

✅ Data cleaned


💾 Saving to: file:///home/suvendu/mlbd/project1/CSL7100-Project/parquet


✅ ETL Completed


In [4]:
#Data Validation (Quick sanity check)

In [5]:
from pyspark.sql import SparkSession
from config import PATHS

spark = SparkSession.builder.getOrCreate()

#df = spark.read.parquet(PATHS["parquet"])
df = spark.read.parquet(PATHS["parquet"])


df.show(5)
df.printSchema()
print("Total records:", df.count())

+----------+--------------+------+----------+--------+----+--------------------+-----------------+
|   item_id|       user_id|rating| timestamp|verified|vote|          reviewText|          summary|
+----------+--------------+------+----------+--------+----+--------------------+-----------------+
|B000VWE5OY|A2YQJ5O24Z4Z94|   4.0|1473465600|    true|   0|On a whole a good...|      Good Series|
|B00C7C02KC|A313UHQDE21KQY|   5.0|1406592000|    true|   0|FRANCIS HA  GREAT!!!|FRANCIS HA YES!!!|
|B00006G8J8|A2XWWT5SFQ0UZL|   4.0|1459209600|    true|   0|            Ver Good|       Four Stars|
|B00D2YCLF8|A1GCCMLWIPEKV2|   5.0|1395532800|    true|  34|Ok , I'm really g...|    Yoga on Crack|
|B00R8GUXPG|A2WAQUY8K56G19|   5.0|1424390400|    true|   0|         great movie|       Five Stars|
+----------+--------------+------+----------+--------+----+--------------------+-----------------+
only showing top 5 rows
root
 |-- item_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |

In [6]:
print("Total records:", df.count())

Total records: 134324


In [7]:
#Basic Statistics
#df.describe().show()

In [8]:
import filter_5core_new
import importlib

importlib.reload(filter_5core_new)

<module 'filter_5core_new' from '/home/suvendu/mlbd/project1/CSL7100-Project/code/filter_5core_new.py'>

In [9]:
from filter_5core_new import run_5core

run_5core()

💻 Running in LOCAL mode
📥 Loading data from: file:///home/suvendu/mlbd/project1/CSL7100-Project/parquet


26/04/21 19:06:57 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Initial records: 134324

🔍 Estimating optimal k...

Testing k=2
  Iteration 1


  Iteration 2


  Iteration 3


...

  Iteration 2
  Iteration 3
k=5 → users=0, items=0, edges=0

✅ Selected optimal k = 2

🔁 Iteration 1


Records before filtering: 134324



🔁 Iteration 2
Records before filtering: 132147



🔁 Iteration 3
Records before filtering: 132108



🔁 Iteration 4
Records before filtering: 132108
✅ Converged
💾 Saving filtered data to: file:///home/suvendu/mlbd/project1/CSL7100-Project/parquet/5core
✅ 5-core filtering completed


In [10]:
from encode_ids import run_encoding

run_encoding()

💻 Running in LOCAL mode
🔢 Encoding user_id and item_id...


26/04/21 19:07:40 WARN DAGScheduler: Broadcasting large task binary with size 1502.0 KiB


✅ ID Encoding Completed


In [11]:
from etl import create_spark_session

spark = create_spark_session()



df = spark.read.parquet(PATHS["parquet"] + "/encoded")

df.show(5)
df.printSchema()

💻 Running in LOCAL mode
+-------+-------+------+----------+--------+----+--------------------+--------------------+
|item_id|user_id|rating| timestamp|verified|vote|          reviewText|             summary|
+-------+-------+------+----------+--------+----+--------------------+--------------------+
|  13803|   3774|   4.0|1422403200|    true|   0|I had fun relivin...|The Tarzan Collec...|
|   2334|    430|   5.0|1484870400|    true|   0|              Great!|          Five Stars|
|  10091|    382|   5.0|1475712000|    true|   0|          GOOD MOVIE|                LIKE|
|   4118|   8366|   5.0|1361836800|    true|   0|Probably best wri...|Top notch, great ...|
|   3639|    592|   5.0|1423440000|    true|   0|One of the best t...|One of the best t...|
+-------+-------+------+----------+--------+----+--------------------+--------------------+
only showing top 5 rows
root
 |-- item_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- rating: double (nullable = true)
 

In [12]:
import graph_builder_new
import importlib

importlib.reload(graph_builder_new)

<module 'graph_builder_new' from '/home/suvendu/mlbd/project1/CSL7100-Project/code/graph_builder_new.py'>

In [13]:
from graph_builder_new import build_graph
from etl import user_based_split
df = spark.read.parquet(PATHS["parquet"] + "/encoded")

train_df, test_df = user_based_split(df)



🔀 Performing user-based split (corrected)
Train: 119550, Test: 12558


In [14]:
import graph_builder_new
import importlib

importlib.reload(graph_builder_new)

<module 'graph_builder_new' from '/home/suvendu/mlbd/project1/CSL7100-Project/code/graph_builder_new.py'>

In [15]:
# SPARK NODE2VEC (Word2Vec-based)
#Node2Vec:- Learns global graph structure - → captures similarity between users/items
# Random Walks + Word2Vec (Spark ML)
# Nodes appearing in similar walks → similar embeddings

In [16]:
!pip install node2vec --trusted-host pypi.org --trusted-host files.pythonhosted.org

In [17]:
# For Node2Vec (Random Walk + Word2Vec)
# STEP 1: Prepare Adjacency List- Generate Random Walks-

In [18]:
from pyspark.sql.functions import col, explode, lit
from pyspark.sql.functions import sum as spark_sum
from pyspark.sql.functions import struct, collect_list, map_from_entries
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.types import MapType, DoubleType
from pyspark.sql.functions import udf

In [19]:
from graph_builder_new import build_graph_node2vec , generate_walks

# Step 1: Build graph  #  Prepare Adjacency List
adj, max_user_id = build_graph_node2vec(train_df)

# Step 2: Generate walks
walks = generate_walks(adj, num_walks=10, walk_length=20) # 

walks_rdd = spark.sparkContext.parallelize(walks)
walks_df = walks_rdd.map(lambda x: (x,)).toDF(["walk"])

# Step 3: Word2Vec- Train Word2Vec
from pyspark.ml.feature import Word2Vec

word2vec = Word2Vec(
    vectorSize=64,  # increased from 64 
    windowSize=10,   # increased from 5
    minCount=1,
    inputCol="walk",
    outputCol="embedding"
)

model = word2vec.fit(walks_df)

# Step 4: Get Node Embeddings
embeddings = model.getVectors() \
    .withColumn("id", col("word").cast("int")) \
    .select("id", col("vector").alias("embedding"))

Max user_id: 19150


Adjacency nodes: 35078


In [20]:
# Split Users and Items (Same as your logic)

In [21]:
user_embeddings = embeddings.filter(col("id") <= max_user_id) \
    .withColumnRenamed("id", "user_id")

item_embeddings = embeddings.filter(col("id") > max_user_id) \
    .withColumn(
        "item_id", col("id") - (max_user_id + 1)
    ).select("item_id", "embedding")

In [22]:
# Dot Product Similarity

In [23]:
import builtins
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

def dot_product(v1, v2):
    if v1 is None or v2 is None:
        return 0.0
    return float(builtins.sum(a*b for a,b in zip(v1, v2)))

dot_udf = udf(dot_product, DoubleType())

In [24]:
# Candidate Filtering

In [25]:
top_items = train_df.groupBy("item_id").count() \
    .orderBy(col("count").desc()) \
    .limit(5000)   # 🔥 reduce from 8000 → faster

candidate_items = item_embeddings.join(top_items, "item_id")

In [26]:
# Generate Recommendations

In [27]:
## Candidate Scoring
recs = user_embeddings.alias("u") \
    .crossJoin(candidate_items.alias("i")) \
    .withColumn(
        "score",
        dot_udf(col("u.embedding"), col("i.embedding"))
    ).select(
        col("u.user_id"),
        col("i.item_id"),
        col("score")
    )
#  REMOVE SEEN ITEMS
seen_items = train_df.select("user_id", "item_id").distinct()

recs = recs.join(
    seen_items,
    ["user_id", "item_id"],
    "left_anti"
)

In [28]:
# Top-K per User

In [29]:
K = 5

window = Window.partitionBy("user_id").orderBy(col("score").desc())

recommendations = recs.withColumn(
    "rnk", row_number().over(window)
).filter(col("rnk") <= K).drop("rnk")

In [30]:
# Evaluate

In [31]:
# Build Ground Truth from test_df

In [32]:
from pyspark.sql.functions import collect_list

ground_truth = test_df.groupBy("user_id").agg(
    collect_list("item_id").alias("actual_items")
)

In [33]:
# Ensure Recommendations Format

In [34]:
recommendations = recommendations.select(
    "user_id", "item_id", col("score").alias("rank")
)

In [35]:
# DEBUG

In [36]:
'''print("Recommendations count:", recommendations.count())
print("Ground truth count:", ground_truth.count())

common_users = recommendations.select("user_id").distinct() \
    .intersect(ground_truth.select("user_id").distinct())

print("Users in both:", common_users.count())'''

'print("Recommendations count:", recommendations.count())\nprint("Ground truth count:", ground_truth.count())\n\ncommon_users = recommendations.select("user_id").distinct()     .intersect(ground_truth.select("user_id").distinct())\n\nprint("Users in both:", common_users.count())'

In [37]:
# Run Evaluation

In [38]:
recommendations = recommendations.repartition(200)
ground_truth = ground_truth.repartition(200)

recommendations.cache()
ground_truth.cache()

DataFrame[user_id: int, actual_items: array<int>]

In [39]:
#spark.conf.set("spark.sql.shuffle.partitions", "200")
#spark.conf.set("spark.executor.memory", "6g")   # increase if possible
#spark.conf.set("spark.driver.memory", "4g")

In [40]:
from evaluation import evaluate_precision_recall, compute_map ,evaluate_precision_recall_safe, compute_map_safe
from pyspark.sql.functions import collect_list

precision, recall = evaluate_precision_recall_safe(
    recommendations,
    ground_truth,
    k=5
)

pred_items_df = recommendations.groupBy("user_id").agg(
    collect_list("item_id").alias("pred_items")
)

#map_k = compute_map_safe(recommendations, ground_truth, k=5)

print("\n🎯 SPARK NODE2VEC RESULTS")
print("Precision@5:", precision)
print("Recall@5:", recall)
#print("MAP@5:", map_k)


🎯 SPARK NODE2VEC RESULTS
Precision@5: 0.19999999999999984
Recall@5: 0.6304050355774493


In [41]:
map_k = compute_map_safe(recommendations, ground_truth, k=5)
print("MAP@5:", map_k)

MAP@5: 0.39078065134099615


In [42]:
user_profiles.printSchema()
item_features.printSchema()

NameError: name 'user_profiles' is not defined

In [43]:
recommendations.printSchema()
recommendations.show(5)

root
 |-- user_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- rank: double (nullable = true)

+-------+-------+------------------+
|user_id|item_id|              rank|
+-------+-------+------------------+
|   8086|    571|1.7389285540443702|
|   1580|   3191|2.0440787648453864|
|  16500|   2185| 3.001877574069606|
|  15003|   1372|1.9845858040088227|
|   1303|   1587|1.5267204072284137|
+-------+-------+------------------+
only showing top 5 rows


In [44]:
recommendations.orderBy("user_id", col("rank").desc()).show(10)

+-------+-------+------------------+
|user_id|item_id|              rank|
+-------+-------+------------------+
|      0|   2571|1.9822786363565332|
|      0|   1670|1.3860439011805443|
|      0|   1594|1.3798118818635534|
|      0|   1292|1.3159852857818644|
|      0|   3885|1.2957450054350725|
|      1|   2971|1.5261128519924692|
|      1|   4245|1.4901024025590988|
|      1|   2218|1.4650493213068057|
|      1|   1369|1.4402310311881452|
|      1|   2244|1.4066475192893844|
+-------+-------+------------------+
only showing top 10 rows


In [45]:
from pyspark.sql.functions import size

ground_truth.select(size("actual_items")).describe().show()

+-------+------------------+
|summary|size(actual_items)|
+-------+------------------+
|  count|              7024|
|   mean|1.7878701594533029|
| stddev|1.4879039810455676|
|    min|                 1|
|    max|                36|
+-------+------------------+



In [46]:
# 

In [47]:
from pyspark.sql.functions import size

ground_truth_filtered = ground_truth.filter(size("actual_items") >= 3)

In [48]:
precision, recall = evaluate_precision_recall_safe(
    recommendations.join(ground_truth_filtered, "user_id"),
    ground_truth_filtered,
    k=5
)

map_k = compute_map_safe(
    recommendations.join(ground_truth_filtered, "user_id"),
    ground_truth_filtered,
    k=5
)

In [49]:
print("\n🎯 SPARK NODE2VEC RESULTS")
print("Precision@5:", precision)
print("Recall@5:", recall)
print("MAP@5:", map_k)


🎯 SPARK NODE2VEC RESULTS
Precision@5: 0.20000000000000004
Recall@5: 0.22271825396825398
MAP@5: 0.12751736111111112
